# Funciones, Entrada/Salida y Procesamiento de Datos en Bash

Bash permite estructurar el código mediante funciones y ofrece un control granular sobre cómo los datos fluyen entre comandos, archivos y dispositivos. Este notebook profundiza en la modularización y el manejo de descriptores de archivos.

## 1. Funciones en Profundidad

A diferencia de otros lenguajes, Bash no tiene argumentos con nombre. Todo se basa en la posición de los parámetros.

In [ ]:
#!/bin/bash

# Definición de función
function mi_log() {
    # Definir variables locales para evitar colisiones
    local nivel="$1"
    local mensaje="$2"
    local fecha=$(date '+%Y-%m-%d %H:%M:%S')

    echo "[$fecha] [$nivel] $mensaje"
}

# Llamada a la función
mi_log "INFO" "Iniciando el proceso de respaldo"
mi_log "ERROR" "No se pudo conectar al servidor remoto"

### Retorno de Valores
Las funciones en Bash no retornan valores de la misma manera que Python o C. Tienen un **estado de salida** (0-255) y para "retornar" datos, suelen imprimir a `stdout`.

In [ ]:
sumar() {
    local res=$(( $1 + $2 ))
    echo "$res" # Esto es el "retorno"
}

# Capturar el resultado usando sustitución de comandos
resultado=$(sumar 10 20)
echo "La suma es: $resultado"

## 2. Redirecciones Avanzadas y Descriptores de Archivos

- `0`: stdin (entrada estándar)
- `1`: stdout (salida estándar)
- `2`: stderr (salida de errores)

In [ ]:
# Redirigir errores a un archivo y salida normal a otro
comando_peligroso > salida.log 2> errores.log

# Redirigir todo al mismo sitio
comando_peligroso &> todo.log

# Descartar errores (enviarlos al agujero negro)
ls /root 2> /dev/null

# Here Documents (Escribir bloques multilínea)
cat << EOF > configuracion.conf
server_name: localhost
port: 8080
timeout: 30
EOF

## 3. Pipes (Tuberías) y Filtros

Las tuberías permiten que la salida de un programa sea la entrada de otro, creando potentes flujos de procesamiento.

In [ ]:
# Listar procesos, buscar 'python', eliminar el comando grep de la lista y contar
ps aux | grep python | grep -v grep | wc -l

# Ordenar un archivo, eliminar duplicados y contar ocurrencias
cat accesos.log | awk '{print $1}' | sort | uniq -c | sort -nr

## 4. Entrada de Usuario e Interactividad

In [ ]:
# Leer entrada con tiempo de espera y máscara para contraseñas
read -p "Usuario: " usuario
read -sp "Contraseña: " password
echo -e "\nAutenticando a $usuario..."

# Menú interactivo sencillo con 'select'
PS3="Seleccione una opción: "
opciones=("Backup" "Restaurar" "Salir")
select opt in "${opciones[@]}"; do
    case $opt in
        "Backup") echo "Iniciando backup...";;
        "Restaurar") echo "Iniciando restauración...";;
        "Salir") break;;
        *) echo "Opción inválida $REPLY";;
    esac
done

## 5. Manejo de señales (Signal Handling)

Permite que el script responda a eventos externos como `Ctrl+C` (SIGINT).

In [ ]:
cleanup() {
    echo -e "\nInterrupción detectada. Limpiando y saliendo..."
    exit 1
}

# Capturar SIGINT y SIGTERM
trap cleanup SIGINT SIGTERM

echo "Script en ejecución. Presiona Ctrl+C para salir."
while true; do sleep 1; done